# Tarea Guiada: Optimización del Umbral (Dataset Banknote Authentication)

**Contexto de Negocio: Detección de Fraude**

Tu estudio de animación ahora recibe pagos de clientes en efectivo físico. Lamentablemente, hay billetes falsos circulando. Trabajaremos con el **Banknote Authentication Dataset**, extrayendo características de imágenes de billetes para predecir cuáles son fraudulentos.

## Paso 1: Carga de Datos

Descarga el dataset de billetes desde internet y cárgalo en un DataFrame.

In [2]:
import pandas as pd
from sklearn.metrics import confusion_matrix

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00267/data_banknote_authentication.txt'
column_names = ['variance', 'skewness', 'curtosis', 'entropy', 'class']

# Usa pd.read_csv para cargar los datos usando la URL y los nombres de columna
df_bank = pd.read_csv(url, header=None, names=column_names)

# Muestra las primeras filas
print("Primeras filas del DataFrame:")
print(df_bank.head())

Primeras filas del DataFrame:
   variance  skewness  curtosis  entropy  class
0   3.62160    8.6661   -2.8073 -0.44699      0
1   4.54590    8.1674   -2.4586 -1.46210      0
2   3.86600   -2.6383    1.9242  0.10645      0
3   3.45660    9.5228   -4.0112 -3.59440      0
4   0.32924   -4.4552    4.5718 -0.98880      0


## Paso 2: Análisis de Correlación

Calcula qué característica está más correlacionada negativamente con la clase (billete falso vs auténtico).

In [3]:
# Calcula la correlación usando df_bank.corr()
correlaciones = df_bank.corr()
print("\nMatriz de Correlación:")
print(correlaciones)

print("\nCorrelación de las características con la clase:")
print(correlaciones['class'])


Matriz de Correlación:
          variance  skewness  curtosis   entropy     class
variance  1.000000  0.264026 -0.380850  0.276817 -0.724843
skewness  0.264026  1.000000 -0.786895 -0.526321 -0.444688
curtosis -0.380850 -0.786895  1.000000  0.318841  0.155883
entropy   0.276817 -0.526321  0.318841  1.000000 -0.023424
class    -0.724843 -0.444688  0.155883 -0.023424  1.000000

Correlación de las características con la clase:
variance   -0.724843
skewness   -0.444688
curtosis    0.155883
entropy    -0.023424
class       1.000000
Name: class, dtype: float64


## Paso 3: Clasificador Heurístico (Iterativo)

Crea un clasificador usando una característica (por ejemplo, 'variance'). Intenta encontrar un umbral manual para detectar la mayor cantidad de fraude sin bloquear demasiado dinero real.

In [4]:
def clasificador_fraude(row):
    # Implementa una lógica basada en 'variance'. Ej: Si variance < 0, es fraude (1)
    if row['variance'] < 0:
        return 1
    else:
        return 0

# Aplica la función y genera la matriz de confusión
y_pred = df_bank.apply(clasificador_fraude, axis=1)
y_true = df_bank['class']

matriz_conf = confusion_matrix(y_true, y_pred)
print("\nMatriz de Confusión del Clasificador Heurístico:")
print(matriz_conf)



Matriz de Confusión del Clasificador Heurístico:
[[656 106]
 [108 502]]


## Evaluación Teórica / Decisión de Negocios

Argumenta matemáticamente: ¿Por qué en la detección de fraude preferimos un modelo con alta Sensibilidad (Recall), incluso si eso significa incrementar un poco los Falsos Positivos?

R.-Definición matemática: La Sensibilidad (Recall o tasa de verdaderos positivos) se define como:$$\text{Recall} = \frac{\text{VP}}{\text{VP} + \text{FN}}$$
donde VP representa los Verdaderos Positivos (fraudes correctamente detectados) y FN representa los Falsos Negativos (fraudes no detectados que pasan como legítimos).

Impacto en el negocio: En un entorno de detección de fraude con efectivo, un Falso Negativo (FN) implica aceptar un billete falso como si fuera real, lo que se traduce en una pérdida financiera directa y un fallo de seguridad para el estudio.

Justificación de la preferencia: Al buscar una alta Sensibilidad, el objetivo es minimizar los Falsos Negativos ($\text{FN} \to 0$), garantizando que la gran mayoría de los billetes fraudulentos sean atrapados. Aunque esto puede aumentar los Falsos Positivos (FP, es decir, rechazar erróneamente algún billete legítimo), desde la perspectiva de gestión de riesgos el costo monetario y de reputación de aceptar fraude financiero directo supera ampliamente la molestia operativa menor de verificar manualmente o rechazar un pago legítimo ocasional.
